In [1]:
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
    roc_auc_score
)
from xgboost import XGBClassifier

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_DIR   = '/home/laura/Scriptie/bewerkte_data/THRawS'
THRESHOLDS = np.linspace(0.05, 0.95, 19)
# ────────────────────────────────────────────────────────────────────────────

In [2]:
X_train = np.load(os.path.join(DATA_DIR, 'X_train_THRawS.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train_THRawS.npy'))
X_test  = np.load(os.path.join(DATA_DIR, 'X_test_THRawS.npy'))
y_test  = np.load(os.path.join(DATA_DIR, 'y_test_THRawS.npy'))

In [3]:
mc_model = DummyClassifier(strategy='prior', random_state=42)
mc_model.fit(X_train, y_train)

mc_pred = mc_model.predict(X_test)
mc_fire_idx = list(mc_model.classes_).index(1)
mc_probs = mc_model.predict_proba(X_test)[:, mc_fire_idx]

print("=== MAJORITY CLASS BASELINE ===")
print(f"Prior probability of fire (class 1): {mc_probs[0]:.4f}")
print(f"Accuracy: {accuracy_score(y_test, mc_pred):.3f}")
print(classification_report(y_test, mc_pred,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

=== MAJORITY CLASS BASELINE ===
Prior probability of fire (class 1): 0.6250
Accuracy: 0.667
              precision    recall  f1-score   support

No event (0)       0.00      0.00      0.00        90
   Event (1)       0.67      1.00      0.80       180

    accuracy                           0.67       270
   macro avg       0.33      0.50      0.40       270
weighted avg       0.44      0.67      0.53       270



In [4]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=100, random_state=42,
    n_jobs=-1, class_weight='balanced'
)
rf.fit(X_train, y_train)
fire_idx_rf = list(rf.classes_).index(1)
rf_probs = rf.predict_proba(X_test)[:, fire_idx_rf]

print("=== RANDOM FOREST (threshold 0.5) ===")
rf_pred_05 = (rf_probs >= 0.5).astype(int)
print(classification_report(y_test, rf_pred_05,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

# XGBoost
xgb_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1,
    scale_pos_weight=xgb_ratio, eval_metric='logloss'
)
xgb.fit(X_train, y_train)
fire_idx_xgb = list(xgb.classes_).index(1)
xgb_probs = xgb.predict_proba(X_test)[:, fire_idx_xgb]

print("=== XGBOOST (threshold 0.5) ===")
xgb_pred_05 = (xgb_probs >= 0.5).astype(int)
print(classification_report(y_test, xgb_pred_05,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

=== RANDOM FOREST (threshold 0.5) ===
              precision    recall  f1-score   support

No event (0)       0.58      0.46      0.51        90
   Event (1)       0.75      0.83      0.79       180

    accuracy                           0.71       270
   macro avg       0.67      0.64      0.65       270
weighted avg       0.70      0.71      0.70       270

=== XGBOOST (threshold 0.5) ===
              precision    recall  f1-score   support

No event (0)       0.50      0.56      0.52        90
   Event (1)       0.76      0.72      0.74       180

    accuracy                           0.66       270
   macro avg       0.63      0.64      0.63       270
weighted avg       0.67      0.66      0.67       270



In [5]:
rf_results, xgb_results = [], []

for t in THRESHOLDS:
    rf_pred  = (rf_probs  >= t).astype(int)
    xgb_pred = (xgb_probs >= t).astype(int)

    rf_results.append({
        'threshold': t,
        'probs':     rf_probs,
        'preds':     rf_pred,
        'f1':        f1_score(y_test, rf_pred, zero_division=0),
        'f1_wtd':    f1_score(y_test, rf_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_test, rf_pred, zero_division=0),
        'recall':    recall_score(y_test, rf_pred, zero_division=0),
        'accuracy':  accuracy_score(y_test, rf_pred),
        'auroc':     roc_auc_score(y_test, rf_probs),
    })
    xgb_results.append({
        'threshold': t,
        'probs':     xgb_probs,
        'preds':     xgb_pred,
        'f1':        f1_score(y_test, xgb_pred, zero_division=0),
        'f1_wtd':    f1_score(y_test, xgb_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_test, xgb_pred, zero_division=0),
        'recall':    recall_score(y_test, xgb_pred, zero_division=0),
        'accuracy':  accuracy_score(y_test, xgb_pred),
        'auroc':     roc_auc_score(y_test, xgb_probs),
    })

print(f"Done — {len(THRESHOLDS)} thresholds evaluated.")

Done — 19 thresholds evaluated.


In [6]:
for name, results in [('Random Forest', rf_results), ('XGBoost', xgb_results)]:
    best = max(results, key=lambda x: x['f1'])
    print(f"=== {name} ===")
    print(f"  Best threshold: {best['threshold']:.2f}")
    print(f"  F1:        {best['f1']:.3f}")
    print(f"  F1 wtd:    {best['f1_wtd']:.3f}")
    print(f"  Precision: {best['precision']:.3f}")
    print(f"  Recall:    {best['recall']:.3f}")
    print(f"  Accuracy:  {best['accuracy']:.3f}")
    print(f"  AUROC:     {best['auroc']:.3f}")
    print()

# Full table for Random Forest
print(f"{'Threshold':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 50)
for r in rf_results:
    print(f"  {r['threshold']:>8.2f} {r['f1']:>8.3f} {r['precision']:>10.3f} "
          f"{r['recall']:>8.3f} {r['accuracy']:>10.3f}")

=== Random Forest ===
  Best threshold: 0.15
  F1:        0.817
  F1 wtd:    0.636
  Precision: 0.701
  Recall:    0.978
  Accuracy:  0.707
  AUROC:     0.815

=== XGBoost ===
  Best threshold: 0.05
  F1:        0.785
  F1 wtd:    0.623
  Precision: 0.695
  Recall:    0.900
  Accuracy:  0.670
  AUROC:     0.740

 Threshold       F1  Precision   Recall   Accuracy
--------------------------------------------------
      0.05    0.805      0.674    1.000      0.678
      0.10    0.813      0.690    0.989      0.696
      0.15    0.817      0.701    0.978      0.707
      0.20    0.805      0.698    0.950      0.693
      0.25    0.800      0.700    0.933      0.689
      0.30    0.792      0.701    0.911      0.681
      0.35    0.792      0.707    0.900      0.685
      0.40    0.793      0.719    0.883      0.693
      0.45    0.791      0.731    0.861      0.696
      0.50    0.792      0.754    0.833      0.707
      0.55    0.792      0.790    0.794      0.722
      0.60    0.800    

In [8]:
print(f"mc_model.classes_: {mc_model.classes_}")
print(f"Prior per klasse: {mc_model.predict_proba(X_test[:1])}")

mc_model.classes_: [0 1]
Prior per klasse: [[0.375 0.625]]


In [5]:
rf_results_cs, xgb_results_cs = [], []

for t in THRESHOLDS:
    #pos_weight = np.clip((1 - t) / t, 0.1, 10)
    pos_weight = np.clip((1 - t) / t, 0.3, 3)

    # Random Forest
    rf_cs = RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1,
        class_weight={0: 1.0, 1: pos_weight}
    )
    rf_cs.fit(X_train, y_train)
    fire_idx = list(rf_cs.classes_).index(1)
    rf_prob_cs = rf_cs.predict_proba(X_test)[:, fire_idx]
    rf_pred_cs = (rf_prob_cs >= 0.5).astype(int)

    rf_results_cs.append({
        'threshold': t,
        'pos_weight': pos_weight,
        'f1':        f1_score(y_test, rf_pred_cs, zero_division=0),
        'precision': precision_score(y_test, rf_pred_cs, zero_division=0),
        'recall':    recall_score(y_test, rf_pred_cs, zero_division=0),
        'accuracy':  accuracy_score(y_test, rf_pred_cs),
        'auroc':     roc_auc_score(y_test, rf_prob_cs),
    })

    # XGBoost
    xgb_cs = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1,
        scale_pos_weight=pos_weight, eval_metric='logloss'
    )
    xgb_cs.fit(X_train, y_train)
    fire_idx_xgb = list(xgb_cs.classes_).index(1)
    xgb_prob_cs = xgb_cs.predict_proba(X_test)[:, fire_idx_xgb]
    xgb_pred_cs = (xgb_prob_cs >= 0.5).astype(int)

    xgb_results_cs.append({
        'threshold': t,
        'pos_weight': pos_weight,
        'f1':        f1_score(y_test, xgb_pred_cs, zero_division=0),
        'precision': precision_score(y_test, xgb_pred_cs, zero_division=0),
        'recall':    recall_score(y_test, xgb_pred_cs, zero_division=0),
        'accuracy':  accuracy_score(y_test, xgb_pred_cs),
        'auroc':     roc_auc_score(y_test, xgb_prob_cs),
    })

print(f"{'Threshold':>10} {'PosWeight':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 65)
for r in rf_results_cs:
    print(f"  {r['threshold']:>8.2f} {r['pos_weight']:>10.3f} {r['f1']:>8.3f} "
          f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['accuracy']:>10.3f}")

 Threshold  PosWeight       F1  Precision   Recall   Accuracy
-----------------------------------------------------------------
      0.05      3.000    0.713      0.689    0.739      0.604
      0.10      3.000    0.713      0.689    0.739      0.604
      0.15      3.000    0.713      0.689    0.739      0.604
      0.20      3.000    0.713      0.689    0.739      0.604
      0.25      3.000    0.713      0.689    0.739      0.604
      0.30      2.333    0.730      0.692    0.772      0.619
      0.35      1.857    0.741      0.707    0.778      0.637
      0.40      1.500    0.736      0.708    0.767      0.633
      0.45      1.222    0.751      0.717    0.789      0.652
      0.50      1.000    0.778      0.713    0.856      0.674
      0.55      0.818    0.779      0.724    0.844      0.681
      0.60      0.667    0.784      0.737    0.839      0.693
      0.65      0.538    0.779      0.732    0.833      0.685
      0.70      0.429    0.800      0.751    0.856      0.715
    